In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import math
import copy

import matplotlib.pyplot as plt
from data_loader import *
from utils import *
from VAE import *
from VAE_joint import *
from combined_data_loader import *
from Optimization_multi_node import *
from Optimization_single_node import *
import pickle

In [ ]:
data=pd.read_csv('../Data/load_data_city_4_2.csv')

In [8]:
class Args:
    def __init__(self):
        # ----- problem / optnet -----
        self.T = 24
        self.base_mva = 100.0
        self.capacity_scale = 4.5
        self.ramp_rate = 0.5
        self.voll = 200.0
        self.vosp = 50.0
        self.M_beta = 1e4
        self.pwl_segments = 10

        # IMPORTANT: add these to match gurobi
        self.reserve_up_ratio = 0.05
        self.reserve_dn_ratio = 0.02
        self.rt_up_ratio = 3.0
        self.rt_dn_ratio = 0.5

        self.N_scen = 20       # <== OptNet真正求解的场景池 (即 K)
        self.S_full = 100       # VAE 现场吐出的大量候选场景数 (S 池)
        self.K_rand = 0       # K里面有多少条纯随机保留(防过拟合)
        self.tau_gumbel = 1.0     # Gumbel Softmax 温度
        self.eps_uniform = 0.1 # 防震荡平滑参数
        self.lambda_div = 1e5   # [新增] 避免多头选到同一个场景的相互排斥惩罚力度

        # ----- 分段训练控制参数 -----
        self.device = "cuda"
       #self.batch_size = 32   # (注意: 原来你的叫 dfl_batch_size, 统一改成 batch_size 给 DataLoader 读)
        self.train_batch_size = 8
        self.test_batch_size = 8
        self.solver = "ECOS"
        
        self.filter_epochs = 5 # Stage 2 (训Filter) 轮数
        self.filter_lr = 1e-3   # Stage 2 学习率
        self.dfl_epochs = 1     # Stage 3 (联合微调) 轮数 (端到端微调极耗时，一般1-3轮即收敛)
        self.dfl_lr = 1e-6      # Stage 3 学习率 (必须极小，防崩坏)
        
args = Args()

In [9]:
set_seed(42)
data_path = "load_data_city_4_2.csv"
target_nodes = [f"4-2-{i}" for i in range(11)]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eps_search=pd.read_csv('../Result/eps_search.csv')
eps=int(eps_search[eps_search['model']=='vae_m']['eps'])
data=pd.read_csv(data_path)
data["DATETIME"] = pd.to_datetime(data["DATETIME"], errors="coerce")
data_2022 = data[data["DATETIME"].dt.year == 2022].copy()
Lmin, Lmax = system_hourly_load_minmax(data_2022, datetime_col="DATETIME",node_cols=target_nodes)
Lmax_total=Lmax.sum(0)# (24,)
Lmin_total=Lmin.sum(0) # (24,)
args.eps_value=eps

In [5]:
models_m, handlers_m, pack_data_m = run_vae_joint(
    data_path=data_path,
    node_cols=target_nodes,
    device=device,
    epochs=1000,
    batch_size=128,
    lr=1e-4,
    patience=50,
    train_length=8760,
    val_ratio=0.2,
    seed=42,
    z_global=16,
    z_node=4,
    hidden=256,
    dropout=0.0,
    beta_g=1e-3,
    beta_n=1e-2,
    beta_anneal=True,
    anneal_warmup=50,
    find_best_ztemp=True,
    z_temp_grid=np.linspace(0.5, 4.0, 8),
    ztemp_search_quantiles=[0.05 * i for i in range(1, 20)],
    ztemp_search_n_samples=50,
    ztemp_search_batch_size=128,
    ckpt_dir="../Model/VAE/ckpt_joint_vae",
)

[ep 0001] train loss=0.963857 recon=0.963720 kl_g=0.2455 kl_n=0.7155 | val loss=1.015648 recon=1.015251 kl_g=0.2336 kl_n=0.6377 beta_g=6e-05 beta_n=0.0006
[ep 0010] train loss=0.744199 recon=0.741377 kl_g=0.8713 kl_n=0.4155 | val loss=0.706855 recon=0.703623 kl_g=0.9571 kl_n=0.4429 beta_g=0.0006 beta_n=0.006
[ep 0020] train loss=0.226879 recon=0.223484 kl_g=1.4232 kl_n=0.1972 | val loss=0.252063 recon=0.249060 kl_g=1.1861 kl_n=0.1817 beta_g=0.001 beta_n=0.01
[ep 0030] train loss=0.115564 recon=0.113085 kl_g=0.4023 kl_n=0.2077 | val loss=0.156038 recon=0.153446 kl_g=0.3912 kl_n=0.2200 beta_g=0.001 beta_n=0.01
[ep 0040] train loss=0.092145 recon=0.089103 kl_g=0.4735 kl_n=0.2569 | val loss=0.132619 recon=0.129111 kl_g=0.5299 kl_n=0.2978 beta_g=0.001 beta_n=0.01
[ep 0050] train loss=0.083391 recon=0.079181 kl_g=1.1213 kl_n=0.3088 | val loss=0.123934 recon=0.119005 kl_g=1.2956 kl_n=0.3633 beta_g=0.001 beta_n=0.01
[ep 0060] train loss=0.079957 recon=0.075532 kl_g=1.9290 kl_n=0.2496 | val los

In [ ]:
set_seed(0)
window_pack_m_vae_train = sample_window_vae_joint(
    models_m, handlers_m, pack_data_m,
    target_nodes=target_nodes,
    horizon_days=292, start_day=0, n_samples=200, seq_len=24,
    z_temp=None, split="train",
)

set_seed(0)
window_pack_m_vae_val = sample_window_vae_joint(
    models_m, handlers_m, pack_data_m,
    target_nodes=target_nodes,
    horizon_days=73, start_day=0, n_samples=200, seq_len=24,
    z_temp=None, split="val",
)

set_seed(0)
window_pack_m_vae_test = sample_window_vae_joint(
    models_m, handlers_m, pack_data_m,
    target_nodes=target_nodes,
    horizon_days=303, start_day=0, n_samples=200, seq_len=24,
    z_temp=None, split="test",
)

In [7]:

import pickle
with open('../Result/VAE/models_m.pkl', 'wb') as f:
    pickle.dump(models_m, f)
with open('../Result/VAE/handlers_m.pkl', 'wb') as f:
    pickle.dump(handlers_m, f)
with open('../Result/VAE/pack_data_m.pkl', 'wb') as f:
    pickle.dump(pack_data_m, f)

with open('../Result/VAE/window_pack_m_vae_test.pkl','wb') as f:
    pickle.dump(window_pack_m_vae_test,f)

with open('../Result/VAE/window_pack_s_vae_train.pkl','wb') as f:
    pickle.dump(window_pack_m_vae_train,f)

with open('../Result/VAE/window_pack_m_vae_val.pkl','wb') as f:
    pickle.dump(window_pack_m_vae_val,f)
